# k4_e5_pipeline — Pipeline Toàn Diện Role C: Chỉ mục mE5, Truy hồi 3 nhánh, Fusion 12 nhánh & Submission

**Thành viên phụ trách:** Role C (mE5-large + Fusion 12 nhánh + Diễn tập)
**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On, Environment = Latest.
**Add Input cần thiết:**
- `mir2-data`: `data/query.parquet`, `data/links_corpus_vi_en_zh.parquet`
- `mir2-chunks`: `chunks/parents/`, `chunks/children/`
- `mir2-qtrans`: bản dịch câu hỏi 3 ngôn ngữ (`qwen25_14b`)
- `mir2-emb-e5`: dataset 121 khối embeddings mE5 (Parents 34 + Children 87 do Role C tạo)
- `mir2-runs`: (tuỳ chọn) 9 nhánh từ Role A & B để chạy Fusion 12 nhánh (`lin12_07` / `rrf12`)

**Dung lượng đĩa:** Chỉ mục FAISS IVF-SQ8 cho mE5 đọc vector trực tiếp từ `/kaggle/input` qua `mmap_mode='r'`,
file index sinh ra chỉ ~12 GB (an toàn 100% dưới hạn mức 20 GB của Kaggle).

In [ ]:
# (1) Khởi tạo môi trường & clone mã nguồn nhánh c-e5
NB_VERSION = 'k4-e5-v2 (23/10/2026)'
USE_TRANS = 'qwen25_14b'
RUN_FUSION = True
RUN_RERANK = True

import glob, os, shutil, sys, time, subprocess
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')
CODE = f'{KAGGLE}/working/code'

os.chdir(KAGGLE)
shutil.rmtree(CODE, ignore_errors=True)
print('Đang clone mã nguồn mới nhất từ GitHub nhánh c-e5...')
cmd_clone = 'git clone --depth 1 -b c-e5 https://github.com/nhantranduynhan11-del/Cross-Lingual-Medical-Retrieval.git ' + CODE
assert subprocess.call(cmd_clone, shell=True) == 0, 'Lỗi clone repository!'

os.chdir(CODE)
if CODE not in sys.path:
    sys.path.insert(0, CODE)

from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks', 'qtrans'), base_config='configs/retrieval_e5.yaml')
print('Setup hoàn tất!')

In [ ]:
# (2) Dựng câu hỏi 3 ngôn ngữ & Xác định đường dẫn emb mE5
for d in locate_all('qtrans'):
    for f in glob.glob(f'{d}/qtrans/*/records*.jsonl'):
        dst = f'{K.work}/qtrans/' + os.path.relpath(f, f'{d}/qtrans')
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy(f, dst)

sh(f'python -m mirret.translate build --config {K.cfg} --use {USE_TRANS}')

# Tìm thư mục emb_e5 trong input
emb_candidates = locate_all('emb_e5') or locate_all('emb')
EMB_E5 = None
for c in emb_candidates:
    if (c / 'emb_e5').exists():
        EMB_E5 = str(c / 'emb_e5')
        break
    if (c / 'parents').exists() and (c / 'children').exists():
        EMB_E5 = str(c)
        break

assert EMB_E5, 'CHƯA TÌM THẤY DATASET EMB_E5! Hãy bấm "+ Add Input" -> Chọn dataset mir2-emb-e5!'
print('Thư mục mE5 Embeddings:', EMB_E5)

In [ ]:
# (3) Lập chỉ mục FAISS IVF-SQ8 cho mE5 (Parents & Children)
# Lưu ý: FAISS chỉ đọc npy qua mmap, file index chỉ tốn ~12GB disk working
for lv in ('parents', 'children'):
    sh(f'python -m mirret.index dense --config {K.cfg} --level {lv} --emb {EMB_E5} --force', f'index_e5_{lv}')

# Mã hoá câu hỏi bằng multilingual-e5-large (tự động đệm tiền tố "query: ")
sh(f'python -m mirret.retrieve qencode --config {K.cfg} --device cuda:0', 'qencode_e5')

In [ ]:
# (4) Truy hồi 3 nhánh mE5: e5_vi, e5_en, e5_zh
sh(f'python -m mirret.retrieve run --config {K.cfg} --prefix e5', 'retrieve_e5')
print('Đã hoàn thành 3 nhánh truy hồi mE5!')

In [ ]:
# (5) Hợp nhất đa nhánh: Linear Fusion (lin12_07) & RRF (rrf12)
# Nạp thêm các run của Role A & B từ mir2-runs (nếu có)
RUNS_DIR = f'{K.work}/runs_e5'
# Dọn dẹp alias trùng tên trong runs_e5 để load_runs nạp đúng BGE-M3 của team
for lg in ('vi', 'en', 'zh'):
    dup = f'{RUNS_DIR}/{lg}_dense.parquet'
    if os.path.exists(dup):
        os.remove(dup)

team_runs = locate('runs', required=False)
all_run_dirs = []
if team_runs:
    all_run_dirs.append(str(team_runs / 'runs'))
    print('Tìm thấy runs của team tại:', team_runs)
all_run_dirs.append(RUNS_DIR)

dirs_arg = ','.join(all_run_dirs)
sh(f'python -m mirret.fuse --config {K.cfg} --run-dirs {dirs_arg}', 'fuse_multi')

In [ ]:
# (6) Stage 2: Rerank với Cross-Encoder (2 GPU) & Tạo bài nộp Submission chuẩn nhóm
if RUN_RERANK:
    # Ưu tiên run lin12_07 nếu có đủ 12 nhánh, nếu không fallback sang rrf6 hoặc lin07
    target_run = 'lin12_07'
    if not os.path.exists(f'{RUNS_DIR}/{target_run}.parquet'):
        for alt in ('lin07', 'rrf12', 'rrf6', 'e5_vi'):
            if os.path.exists(f'{RUNS_DIR}/{alt}.parquet'):
                target_run = alt
                break

    print('Target run cho Rerank:', target_run)
    run_file = f'{RUNS_DIR}/{target_run}.parquet'

    # Rerank song song trên 2 GPU
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.rerank --config {K.cfg} '
         f'--run {run_file} --method cross --top-n 400 --shard {g}/2 --device cuda:0 --out {RUNS_DIR}'
         for g in range(2)], 'rerank_cross')

    # Gộp 2 phần rerank
    sh(f'python -m mirret.rerank --config {K.cfg} --run {run_file} --method cross '
       f'--top-n 400 --shard 0/2 --out {RUNS_DIR}')

    # Sinh bài nộp Submission CHUẨN NHÓM: CỐ ĐỊNH k_doc=200, k_chunk=185 (theo chuẩn configs/retrieval.yaml)
    reranked_file = f'{RUNS_DIR}/{target_run}_cross.parquet'
    if os.path.exists(reranked_file):
        sh(f'python -m mirret.submit --config {K.cfg} --run {reranked_file} --k-doc 200 --k-chunk 185 --name sub_{target_run}_cross')
    sh(f'python -m mirret.submit --config {K.cfg} --run {run_file} --k-doc 200 --k-chunk 185 --name sub_{target_run}')

    # Đồng thời sinh bài nộp mE5 thuần của Role C để đối chiếu độc lập
    sh(f'python -m mirret.submit --config {K.cfg} --run {RUNS_DIR}/e5_vi.parquet --k-doc 200 --k-chunk 185 --name sub_e5_vi')

    report('k4_e5_pipeline')